# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model - HashingVector is same as CountVectorizer but it stores the hash value of the word and not the word itself
# Using binary=True with the CountVectorizer makes "one-hot vectors" which means that
# instead of vector storing the count of each word, it stores 1 if the word is present and 0 if it is not

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [17]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 6167.070, Val Loss: 17563.574


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 11331.206, Val Loss: 17441.523


In [15]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [18]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$93 $41 $8 $18 $39 $171 $4 $29 $20 $71 $327 $178 $76 $188 $13 $9 $12 $16 $82 $75 $38 $1 $89 $76 $273 $254 $337 $40 $47 $40 $30 $135 $33 $14 $113 $127 $70 $130 $112 $66 $162 $51 $29 $65 $78 $57 $78 $94 $18 $30 $13 $46 $98 $39 $176 $29 $33 $132 $12 $50 $69 $11 $19 $23 $404 $115 $42 $176 $15 $239 $12 $18 $124 $87 $13 $76 $52 $74 $30 $41 $81 $121 $18 $25 $15 $75 $38 $132 $98 $129 $26 $30 $29 $6 $34 $68 $48 $48 $133 $193 $2 $53 $1 $55 $12 $50 $102 $303 $11 $22 $16 $67 $110 $21 $23 $128 $153 $73 $70 $29 $15 $349 $50 $22 $66 $55 $28 $171 $36 $25 $28 $99 $112 $36 $89 $32 $80 $64 $56 $57 $27 $149 $5 $201 $178 $89 $52 $307 $83 $7 $21 $164 $22 $43 $9 $119 $154 $2 $45 $14 $106 $1 $19 $34 $467 $6 $50 $33 $22 $44 $11 $34 $204 $22 $26 $8 $27 $14 $35 $77 $441 $7 $178 $23 $9 $111 $70 $36 $27 $13 $104 $34 $66 $12 $7 $67 $53 $5 $3 $8 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [19]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [20]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [21]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [ ]:
# The function for gpt-4.1-mini

def gpt_4__1_mini(item):
    response = completion(model="openai/gpt-4.1-mini", messages=messages_for(item))
    return response.choices[0].message.content

In [27]:
gpt_4__1_mini(test[0])

'$220'

In [28]:
test[0].price

219.0

In [29]:
evaluate(gpt_4__1_mini, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$1 $234 $30 $10 $69 $170 $84 $140 $1 $80 $313 $79 $5 $24 $49 $7 $11 $5 $40 $1 $16 $26 $5 $125 $182 $233 $45 $15 $121 $65 $50 $15 $10 $55 $35 $169 $30 $44 $36 $7 $155 $55 $20 $105 $120 $0 $22 $8 $65 $48 $28 $105 $125 $10 $67 $44 $8 $45 $22 $3 $116 $18 $31 $35 $279 $30 $10 $265 $25 $144 $17 $8 $70 $4 $25 $21 $126 $5 $2 $4 $30 $3 $0 $84 $2 $20 $78 $56 $20 $21 $3 $25 $5 $20 $4 $78 $1 $57 $120 $325 $20 $57 $7 $79 $1 $232 $15 $360 $4 $119 $90 $136 $29 $63 $4 $60 $0 $5 $24 $147 $19 $411 $10 $56 $50 $15 $5 $51 $59 $64 $59 $213 $0 $10 $75 $0 $55 $10 $53 $42 $26 $50 $10 $10 $74 $38 $27 $60 $85 $8 $4 $44 $22 $10 $4 $29 $36 $46 $110 $0 $11 $18 $3 $2 $190 $8 $752 $25 $5 $5 $10 $7 $170 $2 $67 $29 $12 $48 $34 $23 $546 $15 $150 $99 $0 $8 $73 $3 $20 $12 $5 $19 $5 $11 $55 $10 $10 $60 $16 $1 

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-3.1-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [30]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [31]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$0 $74 $5 $10 $30 $130 $64 $75 $8 $70 $87 $20 $3 $14 $49 $8 $21 $20 $10 $29 $26 $16 $5 $5 $12 $223 $196 $3 $61 $65 $10 $30 $0 $55 $25 $119 $30 $38 $64 $18 $130 $40 $17 $65 $40 $5 $7 $2 $65 $48 $26 $110 $245 $0 $27 $44 $3 $40 $33 $2 $116 $53 $32 $65 $80 $0 $50 $305 $15 $54 $17 $2 $80 $4 $20 $16 $25 $2 $2 $7 $0 $3 $11 $74 $13 $10 $78 $76 $10 $16 $3 $10 $5 $10 $2 $83 $1 $82 $100 $275 $30 $13 $7 $10 $19 $132 $12 $360 $2 $129 $30 $115 $9 $63 $54 $10 $7 $0 $14 $417 $6 $110 $0 $6 $0 $5 $0 $31 $19 $64 $104 $7 $10 $5 $35 $2 $15 $10 $111 $2 $6 $350 $25 $8 $14 $8 $15 $160 $5 $8 $4 $94 $28 $60 $1 $101 $31 $36 $45 $5 $139 $17 $8 $1 $60 $2 $452 $20 $5 $3 $10 $2 $120 $13 $69 $11 $5 $27 $4 $18 $134 $15 $250 $59 $17 $3 $53 $17 $20 $5 $5 $9 $12 $61 $50 $50 $10 $10 $21 $9 

In [32]:
def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='none', seed=42)
    return response.choices[0].message.content

In [33]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $114 $5 $20 $10 $135 $74 $95 $3 $100 $63 $120 $5 $9 $39 $3 $11 $20 $40 $39 $76 $4 $5 $25 $72 $203 $66 $3 $121 $62 $13 $30 $30 $57 $5 $169 $60 $37 $24 $11 $160 $30 $15 $5 $70 $5 $9 $4 $75 $58 $26 $111 $225 $0 $27 $11 $6 $80 $88 $5 $106 $45 $29 $30 $120 $0 $80 $285 $15 $54 $16 $2 $40 $0 $25 $18 $56 $2 $3 $3 $10 $4 $11 $74 $7 $25 $78 $46 $40 $13 $3 $15 $5 $5 $1 $68 $1 $92 $95 $195 $40 $23 $3 $9 $49 $142 $15 $340 $4 $89 $20 $266 $4 $8 $54 $30 $1 $1 $24 $247 $14 $29 $10 $24 $10 $15 $5 $31 $1 $49 $149 $28 $10 $0 $95 $2 $5 $10 $22 $2 $26 $99 $15 $7 $36 $28 $10 $60 $65 $8 $0 $114 $22 $60 $5 $41 $46 $37 $30 $5 $39 $17 $2 $1 $190 $1 $751 $20 $10 $2 $5 $3 $160 $10 $59 $41 $3 $27 $36 $16 $4 $15 $220 $39 $10 $8 $53 $12 $10 $3 $5 $19 $10 $41 $50 $50 $10 $40 $21 $6 